# 02 Features: receiver-side account state

Work card 02. Runs `AccountState` over all dense days in time order, so every feature uses only earlier transactions. Fraud labels act as confirmed reports: they flag accounts 24 h later, and only labels from the training period are used.

In [ ]:
import sys, time
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data import load_transactions, drop_sparse_days, time_split
from src.features import FEATURE_NAMES, build_feature_table

## Unit check on hand-made transactions

M receives from A, B, C, sends 200 to X, then receives from D two hours later. A→M is labelled laundering, but the flag only arrives 24 h later.

In [ ]:
t0 = pd.Timestamp('2022-09-01 00:00')
toy = pd.DataFrame({
    'ts': [t0, t0 + pd.Timedelta('10min'), t0 + pd.Timedelta('20min'), t0 + pd.Timedelta('30min'), t0 + pd.Timedelta('2h')],
    'src': ['A', 'B', 'C', 'M', 'D'],
    'dst': ['M', 'M', 'M', 'X', 'M'],
    'amount': [100.0, 50.0, 70.0, 200.0, 10.0],
    'amount_paid': [100.0, 50.0, 70.0, 200.0, 10.0],
    'payment_format': ['ACH'] * 5,
    'receiving_currency': ['US Dollar'] * 5,
    'payment_currency': ['US Dollar'] * 5,
    'label': [1, 0, 0, 0, 0],
})
f, _ = build_feature_table(toy, progress_every=0)

assert f.loc[2, 'dst_in_count_1h'] == 2 and f.loc[2, 'dst_unique_senders_24h'] == 2
assert f.loc[3, 'dst_minutes_since_first_seen'] == -1 and f.loc[3, 'src_out_count_24h'] == 0
assert f.loc[4, 'dst_in_count_1h'] == 0 and f.loc[4, 'dst_in_count_24h'] == 3
assert f.loc[4, 'dst_unique_senders_24h'] == 3 and f.loc[4, 'dst_out_count_24h'] == 1
assert abs(f.loc[4, 'dst_pass_through_24h'] - 200 / 220) < 1e-5
assert f.loc[4, 'dst_flagged'] == 0  # label not yet 'reported'
print('Unit check passed')
f[['dst_in_count_1h', 'dst_in_count_24h', 'dst_unique_senders_24h', 'dst_out_count_24h', 'dst_pass_through_24h', 'dst_flagged']]

## Build features for all dense days

Expect a few minutes. Progress prints every 500,000 rows.

In [ ]:
df = load_transactions(ROOT / 'data' / 'HI-Small_Trans.csv')
dense = drop_sparse_days(df)
train, val, test = time_split(df)
train_end, val_end = train['ts'].max(), val['ts'].max()
del df, train, val, test  # free memory; the split is rebuilt from timestamps below
print(f'Dense rows: {len(dense):,}; training labels used up to {train_end}')

start = time.time()
features, state = build_feature_table(dense, label_cutoff=train_end)
elapsed = time.time() - start
print(f'Built {len(features):,} rows in {elapsed:.0f}s ({1e6 * elapsed / len(features):.1f} µs per transaction)')
print(f'Accounts tracked: {len(state.accounts):,}; flagged accounts: {len(state.flagged):,}')
print(f'Feature table memory: {features.memory_usage(deep=True).sum() / 1e9:.2f} GB')

In [ ]:
ts = dense['ts'].values
features['split'] = np.where(ts <= train_end, 'train', np.where(ts <= val_end, 'val', 'test'))
features['ts'] = ts

features.to_pickle(ROOT / 'data' / 'features_all.pkl')
features.groupby('split', sort=False)['label'].agg(['size', 'sum', 'mean'])

## Leakage check

Features for the first 100,000 rows must be identical whether or not later rows exist.

In [ ]:
head, _ = build_feature_table(dense.iloc[:100_000], label_cutoff=train_end, progress_every=0)
same = np.allclose(head[FEATURE_NAMES].values, features[FEATURE_NAMES].iloc[:100_000].values)
print('No look-ahead:', same)
assert same

## Fraud vs normal (training period)

In [ ]:
tr = features[features['split'] == 'train']
comparison = tr.groupby('label')[FEATURE_NAMES].mean().T
comparison.columns = ['normal_mean', 'laundering_mean']
comparison['ratio'] = comparison['laundering_mean'] / comparison['normal_mean'].replace(0, np.nan)
comparison.sort_values('ratio', ascending=False)

In [ ]:
bands = pd.cut(tr['dst_unique_senders_24h'], [-1, 0, 1, 2, 5, 10, 50, np.inf])
fan_in = tr.groupby(bands, observed=True)['label'].agg(transactions='size', laundering_rate='mean')
fan_in['laundering_rate_%'] = (100 * fan_in.pop('laundering_rate')).round(3)
fan_in

In [ ]:
for col in ['src_flagged', 'dst_flagged', 'sender_known_to_dst_24h', 'cross_currency', 'is_self_transfer']:
    rates = tr.groupby(col)['label'].agg(['size', 'mean'])
    rates['mean'] = (100 * rates['mean']).round(3)
    print(col)
    print(rates.rename(columns={'mean': 'laundering_rate_%'}), '\n')

In [ ]:
try:
    import matplotlib.pyplot as plt
    ax = fan_in['laundering_rate_%'].plot(kind='bar', figsize=(8, 4), color='#0F6E78')
    ax.set_xlabel('Unique senders to the receiver in the last 24 h')
    ax.set_ylabel('Laundering rate (%)')
    ax.set_title('Fan-in vs laundering rate (training period)')
    plt.tight_layout()
    (ROOT / 'screenshots').mkdir(exist_ok=True)
    plt.savefig(ROOT / 'screenshots' / 'fan_in_vs_laundering.png', dpi=150)
    plt.show()
except ImportError:
    print('matplotlib not installed; skip chart')